In [0]:
# === 1) Import the logger ===
from UTILS import ReconciliationLogger
from pyspark.sql import functions as F

## Example for 1 metric by pair of tables

In [0]:
# === 2) Create the logger (one per table pair per job) ===
logger = ReconciliationLogger.ReconciliationLogger(
    spark=spark,
    source_system="EDM",
    target_system="EDM",
    source_table="common_sbx.es088740.mock_employee",
    target_table="common_sbx.es088740.mock_employee2",
    job_run_id="run-123",
    run_mode="SCHEDULED",
)

# === 3) Example metric: row count match ===
src_df = spark.table("common_sbx.es088740.mock_employee")
tgt_df = spark.table("common_sbx.es088740.mock_employee2")

with logger.log_rule(
    rule_name="row_count_match",
    metric_name="COUNT",
    severity="ERROR",
    scope="TABLE",
    tolerance_type="ABS",
    tolerance_value=0.0
) as rule:

    src_cnt = src_df.count()
    tgt_cnt = tgt_df.count()
    diff = src_cnt - tgt_cnt  

    rule.set_values(           
        source_value=str(src_cnt),
        target_value=str(tgt_cnt),
        difference=float(diff),
        status="PASS" if diff == 0 else "FAIL",   # Simple logic for row count match PASS or FAIL
    )

# === 4) End run ===
logger.end_run()

print(f"Reconciliation run completed: {logger.id}")


## Example for multiple metrics by pair of tables

In [0]:
src_df = spark.table("common_sbx.es088740.mock_employee")
tgt_df = spark.table("common_sbx.es088740.mock_employee2")

logger = ReconciliationLogger.ReconciliationLogger(
    spark=spark,
    source_system="EDM",
    target_system="DW",
    source_table=src_df,
    target_table=tgt_df,
    job_run_id="dw_load_2025_01_01",
    run_mode="SCHEDULED",
    notes="Nightly pipeline",
)

# RULE 1 — row count match
with logger.log_rule(
    rule_name="row_count_match",
    metric_name="COUNT",
    severity="ERROR",
    scope="TABLE",
    tolerance_type="ABS",
    tolerance_value=0.0,
) as rule:
    src_cnt = src_df.count()
    tgt_cnt = tgt_df.count()
    diff = src_cnt - tgt_cnt
    rule.set_values(
        source_value=str(src_cnt),
        target_value=str(tgt_cnt),
        difference=float(diff),
        status="PASS" if diff == 0 else "FAIL"
    )

# RULE 2 — max(load_date) difference <= 1 day
with logger.log_rule(
    rule_name="max_hire_date_diff",
    metric_name="MAX",
    scope="COLUMN",
    column_name="hire_date",
    severity="WARN",
    tolerance_type="ABS",
    tolerance_value=1.0,
) as rule:
    src_max = src_df.select(F.max("hire_date")).first()[0]
    tgt_max = tgt_df.select(F.max("hire_date")).first()[0]

    if src_max and tgt_max:
        diff = (src_max - tgt_max).days
    else:
        diff = None

    if diff is None:
        status = "WARN"
    elif abs(diff) <= 1:
        status = "PASS"
    else:
        status = "WARN"

    rule.set_values(
        source_value=str(src_max),
        target_value=str(tgt_max),
        difference=diff,
        status=status,
    )

# RULE 3 — Null rate check on a column
with logger.log_rule(
    rule_name="null_rate_match",
    metric_name="NULL_COUNT",
    scope="COLUMN",
    column_name="employee_id",
    severity="ERROR"
) as rule:
    s_nulls = src_df.filter("employee_id IS NULL").count()
    t_nulls = tgt_df.filter("employee_id IS NULL").count()
    diff = s_nulls - t_nulls

    rule.set_values(
        source_value=str(s_nulls),
        target_value=str(t_nulls),
        difference=float(diff),
        status="PASS" if diff == 0 else "FAIL"
    )

# END RUN
logger.end_run()

print("Reconciliation finished.")
